# SNSデータ解析報告書
## 解析の目的
1．どのような話題についての投稿が多いかを解析。  
2．投稿が、どのような感情で行われているかを解析する。

## データの説明
データの概要：参議院選挙期間中のブルースカイへの投稿の中で、"参政党"という単語が含まれる投稿を収集した。
収集した投稿数：2479

投稿ごとの単語数（形態素解析によるトークン数）の統計は以下の通りです。
- 平均: 58.09
- 標準偏差: 46.71
- 最小: 3
- 最大: 198

## 解析手順
1.テキストデータの抽出  
dataディレクトリに保存されている「参政党」関連のJSONファイル群をすべて読み込む。  
各ファイルから投稿のテキストと著者名を抽出し、一覧のデータ（DataFrame）を作成。

2.話題解析  
投稿内容から、どのような話題が話されているかを自動的に分類・抽出する。

- BERTopic: テキストの内容を元に、似たような投稿をグループ化（クラスタリング）し、それぞれのグループがどのような話題（トピック）について話しているかを抽出する。例えば、「選挙」「投票」に関するグループや、「自民党」「保守」に関するグループなどが特定される。
- LDA (潜在的ディリクレ配分法): もう一つの手法としてLDAを使い、投稿全体に潜む主要な5つの話題を抽出する。それぞれの話題を特徴づける単語群（例：「選挙」「投票」「候補者」など）が出力される。

3.感情分析  
事前学習済みのAIモデル（日本語感情分析モデル）を利用して、一つ一つの投稿が「ポジティブ（肯定的）」「ネガティブ（否定的）」「ニュートラル（中立的）」のいずれに分類されるかを判定。  
分析後、全体の投稿のうち、どの感情の投稿がどれくらいの割合を占めるかを集計し、ランキング形式で表示する。

## 解析
### 1.テキストデータの抽出

In [14]:
import os
import json
import pandas as pd
from tqdm import tqdm
from janome.tokenizer import Tokenizer

# data/の参政党と名のつくファイルを全部読み, そこのtextを抽出.
files = [os.path.join("../data", x) for x in os.listdir("../data") if "参政党" in x and not x.endswith(".pickle")]
all_records = [] # [(author/displayname, record/text)]
for file_path in tqdm(files, desc="JSONファイルの読み込み"):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        for record_item in data:
            if 'record' in record_item and 'text' in record_item['record']:
                # all_records.append(record_item['record']['text'])
                record = ()
                record += (record_item['record']['text'],)
                record += (record_item['author']['displayName'],) if 'author' in record_item and 'displayName' in record_item['author'] else ('',)
                all_records.append(record)

# DataFrameを作成
df = pd.DataFrame(all_records, columns=['text', 'author'])
print(f"抽出されたレコード数: {len(df)}")

# 投稿ごとの単語数を計算
t = Tokenizer()
df['word_count'] = df['text'].apply(lambda x: len([token.surface for token in t.tokenize(x)]))

# 統計量を計算して表示
print("--- 投稿ごとの単語数の統計量 ---")
print(df['word_count'].describe())


JSONファイルの読み込み: 100%|██████████| 6/6 [00:00<00:00, 30.57it/s]


抽出されたレコード数: 2479
--- 投稿ごとの単語数の統計量 ---
count    2479.000000
mean       58.085115
std        46.708734
min         3.000000
25%        22.000000
50%        43.000000
75%        82.000000
max       198.000000
Name: word_count, dtype: float64


### 2.話題解析

In [15]:
from janome.tokenizer import Tokenizer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# テキストの前処理
t = Tokenizer()
df['tokens'] = df['text'].apply(lambda x: [token.surface for token in t.tokenize(x)])
df['tokenized_text'] = df['tokens'].apply(lambda x: ' '.join(x))

# BERTopic
print("--- BERTopic ---")
bertopic_model = BERTopic(language="japanese", verbose=True)
topics, probs = bertopic_model.fit_transform(df['tokenized_text'])

print(bertopic_model.get_topic_info())
for i in range(len(bertopic_model.get_topics()) - 1): # トピック-1はoutlierなので除外
    print(f"Topic {i}: {bertopic_model.get_topic(i)}")

# 潜んでいる話題をLDAで抽出
print("--- Latent Dirichlet Allocation (LDA) ---")
vectorizer = CountVectorizer(max_df=0.95, min_df=2) # 日本語のテキストであるため、stop_words='english' を削除。
dtm = vectorizer.fit_transform(df['tokenized_text'])
lda = LatentDirichletAllocation(n_components=5, random_state=0)
lda.fit(dtm)

def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx}: "
        message += ' '.join([feature_names[i]
                             for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

n_top_words = 10
feature_names = vectorizer.get_feature_names_out()
print_top_words(lda, feature_names, n_top_words)

2025-09-21 15:37:02,322 - BERTopic - Embedding - Transforming documents to embeddings.


--- BERTopic ---


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

2025-09-21 15:37:12,078 - BERTopic - Embedding - Completed ✓
2025-09-21 15:37:12,079 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-21 15:37:17,167 - BERTopic - Dimensionality - Completed ✓
2025-09-21 15:37:17,168 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-21 15:37:17,233 - BERTopic - Cluster - Completed ✓
2025-09-21 15:37:17,240 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-09-21 15:37:17,326 - BERTopic - Representation - Completed ✓


    Topic  Count                      Name  \
0      -1    932            -1_参政_ない_いる_ます   
1       0    242             0_ない_てる_から_けど   
2       1     99            1_自民党_だろ_参政_ない   
3       2     84             2_投票_選挙_自民_今回   
4       3     72           3_参政_シレッ_てめえ_rp   
5       4     54   4_hatelabo_anond_新しい_記事   
6       5     52            5_日本_日本人_若者_投票   
7       6     51             6_外国_輸入_とか_日本   
8       7     50             7_議席_獲得_10_22   
9       8     49         8_淫夢淫_詳しく_参政_トレンド   
10      9     47             9_淫夢_戦国_削る_倒す   
11     10     36     10_kindle_memb_mir_料金   
12     11     34           11_東京_トップ_当選_福岡   
13     12     34   12_過去_blog_archives_988   
14     13     33            13_投票_選挙_当日_当選   
15     14     30       14_nhk_参議院_html_過半数   
16     15     28           15_and_投票_選挙_伸び   
17     16     27            16_憲法_立憲_政権_から   
18     17     27           17_日本_淫夢_so_負ける   
19     18     25         18_返納_歳費_明かし_まさかの   
20     19     25            19_化石_

### 3.感情分析

In [17]:
from transformers import pipeline

# 事前学習済みの日本語感情分析モデルをロード。
# モデルを最初にインストールする必要がある: uv add transformers[ja]
sentiment_analyzer = pipeline("sentiment-analysis", model="christian-phu/bert-finetuned-japanese-sentiment", trust_remote_code=True)

# 感情分析を実行。
# これにはデータフレームのサイズに応じて時間がかかる場合がある。
results = sentiment_analyzer(df['text'].tolist())

# 結果をデータフレームに追加。
df['sentiment_label'] = [result['label'] for result in results]
df['sentiment_score'] = [result['score'] for result in results]

print("感情分析完了。結果の一部を表示:")
print(df[['text', 'sentiment_label', 'sentiment_score']].sample(10))

print("感情ごとの投稿数ランキング:")
print(df['sentiment_label'].value_counts())

# 感情ごとの投稿者数を集計
# 著者名が空のレコードは集計から除外
author_sentiment_counts = df[df['author'] != ''].groupby('sentiment_label')['author'].nunique()

print("--- 感情ごとの投稿者数 ---")
print(author_sentiment_counts)

Device set to use cuda:0


感情分析完了。結果の一部を表示:
                                                   text sentiment_label  \
2223  自らの信念から行動しているとはとても思えない党首と感じた。瞬間的なバズり状態を作り出している...         neutral   
1076  まるで石破さんのファンのように自民党のお問い合わせに送りまくったわよ\nマジで頼むー！後参政...        negative   
2389  参政党以外にもヤバい政党多いのに参政党叩き多すぎてうざいほんま。\n\nあいつら他の政党叩く...        negative   
546   わかりやすいところだと宗教右派信者やスピリチュアルを票田としている参政党は表向きも実態もめち...        negative   
1131   >RP\nとはいえ参院は6年ありますから、仮に参政党ブームは去っても影響を除去するのが厄介です。         neutral   
1166  参政党には繋げられませんか？\nTBSラジオの電波を通して、神谷の言い分を耳にしたいです。 ...        positive   
1330  N党が化けの皮が剥がれたというか飽きられたのと同じように、次の選挙では参政党が飽きられていま...        negative   
379   https://trecome.info/articles/007b7b0a-f5d0-4f...        positive   
1920  しかし自民から割れた維新\n自民をサポートするための参政党\n自民の壺政治を引き継ぐための器...        positive   
2066     参政党の支持者って実在してるんだ。FBに繋がってる人で居てびっくりした。（もう繋がり切った）        negative   

      sentiment_score  
2223         0.939970  
1076         0.941646  
2389         0.996931  
546          0.999034  
1131         0.930011  
1166         

## 考察
### 収集したデータの傾向
単語数の平均は58.09だったが、標準偏差が46.71であり、かなり広範囲に散らばっている。因みに、最小は3単語で最大は198単語である。

### 話題の傾向
- 主要な話題：BERTopicの結果では、最も大きな話題(Topic:451件)は「議席」「躍進」といった単語を含んでおり、選挙結果や議席獲得に関する関心の高さが窺える。
- その他の話題：「ワクチン」「憲法」「スパイ防止法案」などの制作に関する話題も見られる。
- LDAの結果：BERTopicと同様に、「選挙」「議席」「ニュース」といった単語が上位に来ている。しかし、BERTopicほど明確な話題を抽出出来てはいない。

### 感情分析
- 感情の分布：分析の結果、positive（855件）、negative（820件）、neutral（804件）の投稿数が拮抗している。
- 意見の分析：参政党に関する投稿は、肯定的・否定的・中立な感情のものがかなり均等に存在している。
- 感情ごとの投稿者数：
    - negative：427
    - neutral：486
    - positive：466
ネガティブな投稿者よりポジティブな投稿者の数がかなり多いことから、ネガティブな投稿を行う人ほど、参政党に関する投稿を熱心に大量に行っていると考えられる。  
参政党に関するネガティブな投稿を行う場合としては、自民党や社会への負の感情・参政党への負の感情に依るものなどが考えられる。それぞれの感情を強く懐く人ほど、参政党への関心が強いのだと考えた。

### 感想
抽出したデータがどのような意味を持つか考察し、さらに幾つかの予想を検証するために、さらにデータを分析すると、少しずつ収集したデータから情報を得る事が出来た。  
例えば、感情分析では3つの感情が拮抗していたため、それぞれの感情ごとの投降者数の統計を取り、そこからさらに分析を行った。  
ここから自民党や社会への負の感情と、参政党への負の感情の投稿数を分けて測定できれば、さらに詳しく解析できると感じた。